In [ ]:
import pydeck
import pyransac3d
import numpy as np
import pandas as pd
from skimage import io
from skimage.transform import resize
import matplotlib.pyplot as plt

In [ ]:
parameters = {
    "kitti": {
        "fx": 239.92654095,
        "fy": 244.61533405,
        "cx": 204.22929592,
        "cy": 063.34630054,
        "true_h": 1.65,
        "rgb": io.imread("2011_10_03_drive_0034_sync_02/0000000000.jpg"),
        "depth": np.load("2011_10_03_drive_0034_sync_02/0000000000.npy"),
        "mask": resize(np.rot90(np.load("2011_10_03_drive_0034_sync_02/0000000_mask.npy"), k=-1), (128, 416), preserve_range=True).astype(int),
    },
    "our": {
        "fx": 1.93924108e+03 / 1920 * 416,
        "fy": 1.93787557e+03 / 1080 * 234,
        "cx": 9.68723704e+02 / 1920 * 416,
        "cy": 5.00828561e+02 / 1080 * 234,
        "true_h": 1.5,
        "rgb": resize(io.imread("outdoor/0000000.jpg"), (234, 416), preserve_range=True).astype(int),
        "depth": resize(np.load("outdoor/0000000_depth.npy").squeeze(), (234, 416), preserve_range=True),
        "mask": resize(np.flip(np.load("outdoor/0000000_mask.npy")), (234, 416), preserve_range=True).astype(int),
    }
}

In [ ]:
fx, fy, cx, cy, true_h, rgb, depth, mask = parameters["kitti"].values()

plt.imshow(rgb)
plt.show()
plt.imshow(depth, cmap="gray")
plt.show()

if mask is not None:
    plt.imshow(mask)
    plt.show()

In [ ]:
def generate_pointcloud(depth, rgb, mask=None, color=None):
    if mask is None:
        mask = np.ones_like(depth, dtype=int)

    rows, cols = depth.shape
    c, r = np.meshgrid(np.arange(cols), np.arange(rows), sparse=True)

    z = depth
    x = z * (c - cx) / fx
    y = z * (r - cy) / fy
    r, g, b = rgb[..., 0], rgb[..., 1], rgb[..., 2]

    if color is not None:
        r = color[0] * np.ones_like(z, dtype=int)
        g = color[1] * np.ones_like(z, dtype=int)
        b = color[2] * np.ones_like(z, dtype=int)

    pointcloud = np.dstack((x, y, z, r, g, b))

    return pointcloud[(z > 0) & (mask > 0)]

In [ ]:
def visualize_pointcloud(point_cloud):
    df = pd.DataFrame(point_cloud, columns=["x", "y", "z", "r", "g", "b"])
    df[["r", "g", "b"]] = df[["r", "g", "b"]].astype(int)
    target = [df.x.mean(), df.y.mean(), df.z.mean()]

    point_cloud_layer = pydeck.Layer(
        "PointCloudLayer",
        df,
        get_position=["x", "y", "z"],
        get_color=["r", "g", "b"])

    view_state = pydeck.ViewState(target=target, controller=True, zoom=1, rotation_x=-90)
    view = pydeck.View(type="OrbitView", controller=True)

    r = pydeck.Deck(point_cloud_layer, initial_view_state=view_state, views=[view])

    return r

In [ ]:
def camera_height(best_eq):
    a, b, c, d = best_eq
    x, y, z = 0, 0, 0
    
    h = np.abs(a * x + b * y + c * z + d) / np.sqrt(np.square([a, b, c]).sum())
    
    return h

In [ ]:
def generate_pointcloud_for_plane(eq, color=(255, 0, 0), radius=100):
    # eq: Ax+By+Cz+D = 0
    # y = -(Ax+Cz+D)/B

    XX, ZZ = np.meshgrid(np.linspace(-radius, radius, 10),
                         np.linspace(-radius, radius, 10))
    X = XX.ravel()
    Z = ZZ.ravel()
    Y = -(eq[0] * X + eq[2] * Z + eq[3]) / eq[1]
    R = color[0] * np.ones_like(X, dtype=int)
    G = color[1] * np.ones_like(X, dtype=int)
    B = color[2] * np.ones_like(X, dtype=int)

    pointcloud = np.vstack([X, Y, Z, R, G, B]).transpose()

    return pointcloud

In [ ]:
def find_scale(depth, rgb, mask=None, visualize=True):
    pointcloud_segmented = generate_pointcloud(depth, rgb, mask, color=(0, 255, 0))

    best_eq, best_inlaers = pyransac3d.Plane().fit(pointcloud_segmented[..., :3], thresh=0.01, maxIteration=100)

    depth_scale = camera_height(best_eq) / true_h
    if not visualize:
        return depth_scale

    print("Camera height is", camera_height(best_eq))
    print("Scale depth", 1 / depth_scale, "times to maximum depth", depth.max() / depth_scale, "meters")

    pointcloud = generate_pointcloud(depth / depth_scale, rgb)
    pointcloud_segmented = generate_pointcloud(depth / depth_scale, rgb, mask, color=(0, 255, 0))
    best_eq, best_inlaers = pyransac3d.Plane().fit(pointcloud_segmented[..., :3], thresh=0.01, maxIteration=100)
    pointcloud_roadway = generate_pointcloud_for_plane(best_eq, color=(255, 0, 0), radius=depth.max() / depth_scale)
    pointcloud_best_inlaers = pointcloud_segmented[best_inlaers]
    pointcloud_best_inlaers[..., 3:] = (0, 0, 255)

    r = visualize_pointcloud(np.vstack([pointcloud, pointcloud_roadway, pointcloud_best_inlaers]))
    r.to_html("temp.html", open_browser=True)

    return depth_scale

In [ ]:
find_scale(depth, rgb, mask, visualize=True)